# 🦀 Day 4: tokio::select! and tokio::join!

Today: run futures concurrently, race them, and add timeouts.

---

## tokio::join! — Wait for All

`tokio::join!` runs multiple futures concurrently and waits for **all** of them to complete. Returns a tuple of results.

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

In [ ]:
use tokio::runtime::Runtime;
use tokio::time::{sleep, Duration};

let rt = Runtime::new().unwrap();
rt.block_on(async {
    let (a, b, c) = tokio::join!(
        async { sleep(Duration::from_millis(50)).await; 1 },
        async { sleep(Duration::from_millis(30)).await; 2 },
        async { sleep(Duration::from_millis(70)).await; 3 }
    );
    println!("join! results: {}, {}, {}", a, b, c);
});

## tokio::try_join! — With Error Handling

Like `join!` but short-circuits on first `Err`. All futures must return `Result`.

In [ ]:
async fn fetch(id: u32) -> Result<String, &'static str> {
    sleep(Duration::from_millis(20)).await;
    if id == 2 { Err("failed") } else { Ok(format!("data-{}", id)) }
}

let rt = Runtime::new().unwrap();
let result = rt.block_on(async {
    tokio::try_join!(fetch(1), fetch(2), fetch(3))
});
match result {
    Ok((a, b, c)) => println!("Success: {}, {}, {}", a, b, c),
    Err(e) => println!("First error: {}", e),
}

## tokio::select! — First One Wins

`tokio::select!` runs multiple futures and completes when the **first** one finishes. The others are cancelled.

In [ ]:
let rt = Runtime::new().unwrap();
rt.block_on(async {
    let result = tokio::select! {
        v = async { sleep(Duration::from_millis(100)).await; "slow" } => v,
        v = async { sleep(Duration::from_millis(20)).await; "fast" } => v,
    };
    println!("First to finish: {}", result);
});

## tokio::time::timeout — Add Deadlines

Wrap any future with `timeout(duration, future)` to get `Ok(result)` or `Err(Elapsed)` if it takes too long.

In [ ]:
use tokio::time::timeout;

let rt = Runtime::new().unwrap();
rt.block_on(async {
    // Completes in time
    let r: Result<_, _> = timeout(
        Duration::from_millis(100),
        async { sleep(Duration::from_millis(20)).await; "done" }
    ).await;
    println!("With enough time: {:?}", r);
    
    // Times out
    let r: Result<_, _> = timeout(
        Duration::from_millis(20),
        async { sleep(Duration::from_millis(100)).await; "done" }
    ).await;
    println!("Timed out: {:?}", r);
});

## Practical Patterns

- **Request with timeout:** wrap your request future in `timeout(...)`
- **First response wins:** use `select!` to race multiple requests
- **Event loop:** `select!` in a loop to process multiple event sources

In [ ]:
// select! in a loop — process until one branch signals done
let rt = Runtime::new().unwrap();
rt.block_on(async {
    let mut count = 0u32;
    loop {
        tokio::select! {
            _ = sleep(Duration::from_millis(30)) => {
                count += 1;
                println!("Tick {}", count);
                if count >= 3 { break; }
            }
        }
    }
    println!("Done after {} ticks", count);
});

## 🏋️ Exercise

In [ ]:
// Exercise: Implement a function that runs an async operation with a timeout.
// If it completes in time, return Ok(result). If it times out, return Err("timeout").
// Use tokio::time::timeout.

// YOUR CODE HERE

## 🎯 Key Takeaways

1. **tokio::join!** — run futures concurrently, wait for all, get tuple of results
2. **tokio::try_join!** — same, but short-circuits on first `Err`
3. **tokio::select!** — race futures, first one wins, others cancelled
4. **tokio::time::timeout** — add a deadline; returns `Err(Elapsed)` on timeout
5. Combine these for robust async patterns: timeouts, first-response-wins, event loops

---
**Tomorrow:** Async channels for message passing between tasks! 📨